In [1]:
# Cell 1：載入所有套件 + 您的模型
import torch
import torch.nn as nn
import numpy as np
import cv2
import mediapipe as mp
from pathlib import Path
import json
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from IPython.display import display, HTML, clear_output
import time

# 您的 encoder（剛訓練好的）
class MotionEncoder(nn.Module):
    def __init__(self, input_dim=177, hidden_dim=256, embed_dim=256, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim*2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, embed_dim)
        )
    def forward(self, x):
        out, (h, c) = self.lstm(x)
        emb = torch.cat([h[-2], h[-1]], dim=-1)
        return self.proj(emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = MotionEncoder().to(device)
encoder.load_state_dict(torch.load("weights/lstm_encoder_best.pth", map_location=device))
encoder.eval()

# 正規化參數
mean = np.load("data/segments/mean.npy")
std  = np.load("data/segments/std.npy")

print("Your ballet semantic encoder has finished loading!")

Your ballet semantic encoder has finished loading!


C:\Users\AW'z\AppData\Local\Temp\ipykernel_24572\1967254137.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder.load_state_dict(torch.load("weights/lstm_encoder_bes

In [2]:
# Cell 2（English Ballet Poet Edition）
import random

poem_templates_en = [
    "She lifts her left leg into arabesque, the toe a silver comet frozen at its zenith against the night.",
    "Arms unfurl like swan's wings, tracing silent verses across the breathless air.",
    "Turn, turn, and turn again-only the tutu and heartbeat remain in the spinning world.",
    "She suspends herself in three seconds of eternity; the universe holds its breath.",
    "A single pointe kisses the floor, shattering the chains of time.",
    "Her arms become moonlight, caressing the trembling skin of the air.",
    "Spine straight as pine, then suddenly blooming into a white lily mid-air.",
    "She leaps into the dream and never wakes again.",
    "In perfect fifth, she writes infinity with her feet.",
    "The body remembers what the soul has always known.",
    "Each extension is a love letter written to gravity-and gravity never replies.",
    "She is the pause between notes where music becomes visible."
]

def generate_poem(tag="graceful extension"):
    poem = random.choice(poem_templates_en)
    return poem + f"\n\n- When you perform \"{tag}\""

print("English Ballet AI Poet is ready!")
print("Test poem:")
print(generate_poem("Left Leg: arabesque"))

English Ballet AI Poet is ready!
Test poem:
Each extension is a love letter written to gravity—and gravity never replies.

— When you perform "Left Leg: arabesque"


In [3]:
# Cell 3
import numpy as np
from sklearn.cluster import KMeans
import torch

# 1. 現場用您所有的 segment 產生 embedding
print("Generating all action embeddings using the encoder you just trained...")
all_embs = []

encoder.eval()
with torch.no_grad():
    for pt_file in Path("data/segments").glob("seg_*.pt"):
        seg = torch.load(pt_file).unsqueeze(0).to(device)  # (1,60,177)
        emb = encoder(seg).cpu().numpy()                  # (1,256)
        all_embs.append(emb)
all_embs = np.concatenate(all_embs, axis=0)  # (N,256)
print(f"Successfully generated {len(all_embs)} action embeddings!")

# 2. 現場訓練 36 類 KMeans（6 部位 × 6 程度）
print("Training 36 types of action clustering...")
kmeans = KMeans(n_clusters=36, random_state=42, n_init=10)
kmeans.fit(all_embs)
print("Clustering complete!")

# 3. 專業芭蕾中文標籤表（6部位 × 6程度）
parts = ["Head", "Left Arm", "Right Arm", "Left Leg", "Right Leg", "Torso"]
levels = ["Still", "Slight", "Medium", "Fast", "Spin", "High/Arabesque"]

def predict_action_tag(embedding):
    cluster_id = kmeans.predict(embedding.cpu().numpy())[0]
    part_idx = cluster_id % 6
    level_idx = cluster_id // 6
    return f"{parts[part_idx]}：{levels[level_idx]}"

# 測試一下
test_emb = torch.randn(1, 256).to(device)
print("Test tag: ", predict_action_tag(test_emb))

print("The motion tagging system training is complete! From now on, there will be no more missing records!")

Generating all action embeddings using the encoder you just trained...


C:\Users\AW'z\AppData\Local\Temp\ipykernel_24572\3071894871.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  seg = torch.load(pt_file).unsqueeze(0).to(device)  # (1,60,1

Successfully generated 672 action embeddings!
Training 36 types of action clustering...


C:\Users\AW'z\.conda\envs\code-lab\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


Clustering complete!
Test tag:  Head：Slight
The motion tagging system training is complete! From now on, there will be no more missing records!


In [4]:
# Cell 4
import numpy as np
import torch

# 您的 compute_features（177 維）
def compute_features(frames: np.ndarray) -> np.ndarray:
    T = frames.shape[0]
    pelvis = (frames[:, 23] + frames[:, 24]) / 2.0
    rel_pos = frames - pelvis[:, None, :]
    rel_flat = rel_pos.reshape(T, -1)
    
    vel = np.zeros_like(frames)
    vel[1:] = frames[1:] - frames[:-1]
    speed = np.linalg.norm(vel, axis=2)
    acc = np.zeros_like(speed)
    acc[1:] = speed[1:] - speed[:-1]
    
    def angle_between(v1, v2):
        v1_norm = v1 / (np.linalg.norm(v1, axis=1, keepdims=True) + 1e-8)
        v2_norm = v2 / (np.linalg.norm(v2, axis=1, keepdims=True) + 1e-8)
        return np.arccos(np.clip(np.sum(v1_norm * v2_norm, axis=1), -1.0, 1.0))
    
    angles = np.stack([
        angle_between(frames[:,11]-frames[:,13], frames[:,15]-frames[:,13]),
        angle_between(frames[:,12]-frames[:,14], frames[:,16]-frames[:,14]),
        angle_between(frames[:,13]-frames[:,11], frames[:,23]-frames[:,11]),
        angle_between(frames[:,14]-frames[:,12], frames[:,24]-frames[:,12]),
        angle_between(frames[:,23]-frames[:,25], frames[:,27]-frames[:,25]),
        angle_between(frames[:,24]-frames[:,26], frames[:,28]-frames[:,26]),
        angle_between(frames[:,25]-frames[:,23], frames[:,11]-frames[:,23]),
        angle_between(frames[:,26]-frames[:,24], frames[:,12]-frames[:,24]),
        angle_between(frames[:,11]-frames[:,12], frames[:,23]-frames[:,12]),
    ], axis=1)
    
    left_arm_speed  = np.linalg.norm(vel[:, [11,13,15]], axis=2).sum(axis=1)
    right_arm_speed = np.linalg.norm(vel[:, [12,14,16]], axis=2).sum(axis=1)
    symmetry = np.abs(left_arm_speed - right_arm_speed)
    energy   = speed.sum(axis=1)
    torso_vec = frames[:,12] - frames[:,11]
    torso_yaw = np.arctan2(torso_vec[:,1], torso_vec[:,0])
    
    features = np.concatenate([
        rel_flat, speed, acc, angles,
        symmetry[:,None], energy[:,None], torso_yaw[:,None]
    ], axis=1).astype(np.float32)
    return features

# 正規化
def normalize_features(feats):
    return (feats - mean) / (std + 1e-8)

# 滑動窗口（關鍵修正！）
buffer = []  # 存 (177,) 的 torch.Tensor

def process_frame_landmarks(landmarks):
    global buffer
    
    # 1. 轉成 (33,3)
    pts = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark])
    
    # 2. 提取單幀特徵 → (1,177)
    feats = compute_features(pts[None, ...])      # (1,177)
    feats = normalize_features(feats)             # (1,177)
    
    # 3. 轉成 (177,) 的 Tensor（關鍵！去掉 batch 維度）
    feats_tensor = torch.from_numpy(feats[0]).to(device)  # ← 這裡是 [0]！
    
    # 4. 存進 buffer
    buffer.append(feats_tensor)
    
    # 5. 維持 60 幀
    if len(buffer) > 60:
        buffer.pop(0)
    
    # 6. 滿 60 幀 → 輸出 embedding
    if len(buffer) == 60:
        seq = torch.stack(buffer)          # (60,177)
        seq = seq.unsqueeze(0).to(device)  # (1,60,177) ← 正確 3D！
        with torch.no_grad():
            emb = encoder(seq)             # (1,256)
        return emb
    return None

print("Real-time motion recognition engine!")

Real-time motion recognition engine!


In [5]:
# Cell 5
import cv2
import mediapipe as mp
from pathlib import Path
from IPython.display import display, HTML, clear_output
import time

# ←←← 您的影片路徑 ←←←
VIDEO_PATH = "./data/test01.mp4"

# 初始化 MediaPipe（一定要有這段！）
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

if not Path(VIDEO_PATH).exists():
    print(f"Video not found: {VIDEO_PATH}")
else:
    print(f"Video found! Ready to play: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)
POEM_INTERVAL = 90        # 每90幀寫一首新詩（約3秒）
frame_count = 0
last_poem_frame = 0

print("Start playing the video; AI is writing a poem for you… (Press q to stop)")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("The video has finished playing! The AI ​​has already written the poem for the entire dance!")
        break
    
    frame_count += 1
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)
    
    if results.pose_landmarks:
        # 畫骨架
        mp_drawing.draw_landmarks(
            frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0,255,0), thickness=2),
            mp_drawing.DrawingSpec(color=(0,0,255), thickness=2)
        )
        
        # 提取動作
        emb = process_frame_landmarks(results.pose_landmarks)
        if emb is not None:
            tag = predict_action_tag(emb)
            
            # 畫面上顯示動作標籤（這次用正確的字型！）
            cv2.putText(frame, tag, (30, 80), 
                       cv2.FONT_HERSHEY_DUPLEX, 1.5, (255, 255, 0), 4, cv2.LINE_AA)
            
            # 每隔一段時間生成新詩
            if frame_count - last_poem_frame >= POEM_INTERVAL:
                poem = generate_poem(tag)
                last_poem_frame = frame_count
                clear_output(wait=True)
                display(HTML(f"""
                <div style="background:rgba(0,0,0,0.9); color:#fff; padding:30px; border-radius:20px; font-family:標楷體,serif; max-width:900px; margin:20px;">
                    <h2 style="color:#ff6b6b; margin-bottom:20px;">當前動作：{tag}</h2>
                    <p style="font-size:26px; line-height:2.4; white-space:pre-wrap;">{poem}</p>
                    <div style="text-align:right; color:#ccc; font-size:18px; margin-top:30px;">
                        -- AI 舞蹈詩人 · 實時生成
                    </div>
                </div>
                """))

    # 放大顯示更好看
    display_frame = cv2.resize(frame, (1280, 720))
    cv2.imshow('Ballet AI Poet - Writing poetry for your dance (press q to end)', display_frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 釋放資源
cap.release()
cv2.destroyAllWindows()
pose.close()
print("Perfect ending!")

The video has finished playing! The AI ​​has already written the poem for the entire dance!
Perfect ending!
